# OptiCell **Stage 3** — multi-backend orchestration + TRA (Colab GPU)

**Goal:** make the pipeline bulletproof by running **threshold / hybrid / Cellpose-SAM** on every CTC sequence, auto-selecting a backend with **auditable rules**, then scoring **TRA/DET/LNK** per backend.

| Piece | Detail |
|-------|--------|
| Data | All CTC training seqs: GOWT1, SIM+, HeLa (01+02) |
| Backends | `threshold`, `hybrid`, `cellpose` (`cpsam`, **GPU**) |
| Select | higher mean FOV confidence → lower count CV → fewer low-conf FOVs |
| TRA | `traccuracy` CTCMetrics after CTC RES export |

**Runtime:** Colab **GPU** (T4/A100). Cellpose model download ~1 GB once.

Policy: measured numbers only. Failures stay in the table.

## 0. Runtime check + install

In [ ]:
import torch, subprocess, sys
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU — Cellpose will be slow; Runtime → Change runtime type → GPU')

In [ ]:
from pathlib import Path
import os, json, zipfile, urllib.request, shutil, importlib

USE_DRIVE = True  # recommended for long multi-backend runs
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/opticell_stage3')
else:
    BASE = Path('/content/opticell_stage3')
BASE.mkdir(parents=True, exist_ok=True)
print('BASE', BASE)

REPO = Path('/content/Virelion-OptiCell')
if not REPO.exists():
    !git clone https://github.com/Virelion-Biotech/Virelion-OptiCell.git
else:
    !git -C /content/Virelion-OptiCell fetch origin main
    !git -C /content/Virelion-OptiCell reset --hard origin/main

%cd /content/Virelion-OptiCell
!pip install -e ".[cellpose]" -q || pip install -e . cellpose -q
!pip install -q traccuracy tifffile packaging

sys.path.insert(0, str(REPO / 'scripts'))
import export_ctc_res as _e
importlib.reload(_e)
from export_ctc_res import export_ctc_res

from traccuracy import run_metrics
from traccuracy.loaders import load_ctc_data
from traccuracy.matchers import CTCMatcher
from traccuracy.metrics import CTCMetrics
print('git', subprocess.check_output(['git','rev-parse','--short','HEAD'], text=True).strip())

## 1. Dataset catalog

In [ ]:
DATASETS = {
    'Fluo-N2DH-GOWT1': {
        'url': 'https://data.celltrackingchallenge.net/training-datasets/Fluo-N2DH-GOWT1.zip',
        'mb': 53, 'sequences': ['01', '02'],
    },
    'Fluo-N2DH-SIM+': {
        'url': 'https://data.celltrackingchallenge.net/training-datasets/Fluo-N2DH-SIM+.zip',
        'mb': 91, 'sequences': ['01', '02'],
    },
    'Fluo-N2DL-HeLa': {
        'url': 'https://data.celltrackingchallenge.net/training-datasets/Fluo-N2DL-HeLa.zip',
        'mb': 182, 'sequences': ['01', '02'],
    },
}

BACKENDS = ['threshold', 'hybrid', 'cellpose']  # full Stage-3 sweep
TRACK_DIST = 50.0
TRACK_GAP = 1
MAX_FRAMES = 0  # 0 = all frames
USE_GPU = True

CTC_ROOT = BASE / 'ctc'
CTC_ROOT.mkdir(parents=True, exist_ok=True)

def download_and_extract(name: str) -> Path:
    meta = DATASETS[name]
    dest = CTC_ROOT / name
    zpath = CTC_ROOT / f'{name}.zip'
    if dest.exists() and any(dest.iterdir()):
        print('[skip]', dest)
        return dest
    print(f'[download] {name} (~{meta["mb"]} MB)')
    urllib.request.urlretrieve(meta['url'], zpath)
    with zipfile.ZipFile(zpath, 'r') as zf:
        zf.extractall(CTC_ROOT)
    if not dest.exists():
        raise FileNotFoundError(dest)
    return dest

def find_seq_gt(root: Path, sequence: str):
    seq = root / sequence
    gt_tra = root / f'{sequence}_GT' / 'TRA'
    man = gt_tra / 'man_track.txt'
    if not seq.is_dir() or not man.is_file():
        raise FileNotFoundError((seq, man))
    return seq, gt_tra, man

print('backends', BACKENDS)

## 2. Orchestrate + TRA helpers

In [ ]:
def score_tra(gt_tra: Path, man: Path, res_dir: Path, name: str) -> dict:
    gt = load_ctc_data(str(gt_tra), str(man), name=f'{name}_GT')
    pred = load_ctc_data(str(res_dir), str(res_dir / 'res_track.txt'), name=f'{name}_RES')
    results, _ = run_metrics(
        gt_data=gt, pred_data=pred,
        matcher=CTCMatcher(), metrics=[CTCMetrics()],
    )
    block = results[0] if isinstance(results, list) else results
    if hasattr(block, 'results'):
        return dict(block.results)
    if isinstance(block, dict) and 'results' in block:
        return dict(block['results'])
    return dict(block)

def orchestrate_sequence(dataset: str, sequence: str, backends=None,
                         max_frames: int = 0, reuse: bool = True):
    backends = backends or BACKENDS
    root = download_and_extract(dataset)
    seq_dir, gt_tra, man = find_seq_gt(root, sequence)
    tag = f'{dataset}_{sequence}'
    s3_root = BASE / 'stage3' / tag
    s3_root.mkdir(parents=True, exist_ok=True)

    # 1) multi-backend Stage-2 via orchestrator script
    orch_out = s3_root / 'orchestrate'
    decision_path = orch_out / 'stage3_decision.json'
    if not (reuse and decision_path.is_file()):
        cmd = [
            sys.executable, str(REPO / 'scripts' / 'run_stage3_orchestrate.py'),
            str(seq_dir), '-o', str(orch_out),
            '--backends', ','.join(backends),
            '--enable-tracking',
            '--track-max-distance', str(TRACK_DIST),
            '--track-max-gap', str(TRACK_GAP),
            '--cellpose-model', 'cpsam',
        ]
        if USE_GPU:
            cmd.append('--gpu')
        if max_frames > 0:
            cmd += ['--max-images', str(max_frames)]
        print('ORCH', ' '.join(cmd[-12:]))
        r = subprocess.run(cmd, capture_output=True, text=True)
        print((r.stdout or '')[-4000:])
        if r.returncode != 0:
            print((r.stderr or '')[-3000:])
            raise RuntimeError(f'orchestrate failed {r.returncode}')
    else:
        print('[reuse orchestrate]', decision_path)

    decision = json.loads(decision_path.read_text())

    # 2) TRA for each successful backend
    tra_rows = []
    for brow in decision.get('per_backend', []):
        backend = brow['backend']
        if brow.get('mean_confidence', -1) < 0 or brow.get('error'):
            tra_rows.append({'dataset': dataset, 'sequence': sequence, 'backend': backend,
                             'error': brow.get('error', 'failed'), 'TRA': None})
            continue
        stage2_dir = Path(brow['out_dir'])
        res_dir = s3_root / 'res' / backend
        print(f'--- export+TRA {tag} / {backend} ---')
        if res_dir.exists():
            shutil.rmtree(res_dir)
        try:
            export_ctc_res(stage2_dir, res_dir)
            metrics = score_tra(gt_tra, man, res_dir, f'{tag}_{backend}')
            row = {
                'dataset': dataset, 'sequence': sequence, 'backend': backend,
                'mean_confidence': brow.get('mean_confidence'),
                'count_cv': brow.get('count_cv'),
                'n_tracks': brow.get('n_tracks'),
                'TRA': metrics.get('TRA'), 'DET': metrics.get('DET'), 'LNK': metrics.get('LNK'),
                'AOGM': metrics.get('AOGM'),
                'fn_nodes': metrics.get('fn_nodes'), 'fp_nodes': metrics.get('fp_nodes'),
                'fn_edges': metrics.get('fn_edges'), 'fp_edges': metrics.get('fp_edges'),
                'ns_nodes': metrics.get('ns_nodes'), 'ws_edges': metrics.get('ws_edges'),
            }
        except Exception as exc:
            row = {'dataset': dataset, 'sequence': sequence, 'backend': backend, 'error': str(exc),
                   'TRA': None}
        print(json.dumps({k: row[k] for k in row if k in ('backend','TRA','DET','LNK','error')}, indent=2))
        tra_rows.append(row)
        out_j = BASE / 'scores' / f'{tag}_{backend}_tra.json'
        out_j.parent.mkdir(parents=True, exist_ok=True)
        out_j.write_text(json.dumps({'summary': row}, indent=2, default=str))

    # 3) compare auto-select vs best TRA (measured post-hoc)
    valid = [r for r in tra_rows if r.get('TRA') is not None]
    best_tra = max(valid, key=lambda r: r['TRA']) if valid else None
    selected = decision.get('decision', {}).get('selected_backend')
    selected_row = next((r for r in tra_rows if r['backend'] == selected), None)
    summary = {
        'dataset': dataset, 'sequence': sequence,
        'auto_selected_backend': selected,
        'auto_selected_TRA': None if not selected_row else selected_row.get('TRA'),
        'best_TRA_backend': None if not best_tra else best_tra['backend'],
        'best_TRA': None if not best_tra else best_tra['TRA'],
        'selection_matched_best_TRA': (
            bool(best_tra and selected_row and best_tra['backend'] == selected_row['backend'])
        ),
        'decision_reason': decision.get('decision', {}).get('reason'),
        'per_backend_tra': tra_rows,
    }
    (s3_root / 'stage3_sequence_summary.json').write_text(json.dumps(summary, indent=2, default=str))
    print('=== SEQUENCE SUMMARY', tag, '===')
    print(json.dumps({k: summary[k] for k in summary if k != 'per_backend_tra'}, indent=2))
    return summary

print('helpers ready')

## 3. Run all sequences (one cell each)

Order: GOWT1 → SIM+ → HeLa. Each cell runs **3 backends** + TRA. Expect hours on T4 for full Cellpose.

Tip: set `BACKENDS = ['threshold','cellpose']` to skip hybrid if time-constrained.

In [ ]:
# --- 1/6 GOWT1 01 ---
s1 = orchestrate_sequence('Fluo-N2DH-GOWT1', '01', max_frames=MAX_FRAMES, reuse=True)

In [ ]:
# --- 2/6 GOWT1 02 ---
s2 = orchestrate_sequence('Fluo-N2DH-GOWT1', '02', max_frames=MAX_FRAMES, reuse=True)

In [ ]:
# --- 3/6 SIM+ 01 ---
s3 = orchestrate_sequence('Fluo-N2DH-SIM+', '01', max_frames=MAX_FRAMES, reuse=True)

In [ ]:
# --- 4/6 SIM+ 02  (threshold collapsed here in Stage-2; Cellpose is the test) ---
s4 = orchestrate_sequence('Fluo-N2DH-SIM+', '02', max_frames=MAX_FRAMES, reuse=True)

In [ ]:
# --- 5/6 HeLa 01 ---
s5 = orchestrate_sequence('Fluo-N2DL-HeLa', '01', max_frames=MAX_FRAMES, reuse=True)

In [ ]:
# --- 6/6 HeLa 02 ---
s6 = orchestrate_sequence('Fluo-N2DL-HeLa', '02', max_frames=MAX_FRAMES, reuse=True)

## 4. Aggregate Stage-3 table

In [ ]:
import pandas as pd

tra_rows = []
seq_rows = []
for p in sorted((BASE / 'scores').glob('*_tra.json')):
    tra_rows.append(json.loads(p.read_text())['summary'])
for p in sorted((BASE / 'stage3').glob('*/stage3_sequence_summary.json')):
    d = json.loads(p.read_text())
    seq_rows.append({k: d[k] for k in d if k != 'per_backend_tra'})

tra_df = pd.DataFrame(tra_rows)
seq_df = pd.DataFrame(seq_rows)
print('=== PER BACKEND TRA ===')
display(tra_df)
print('=== AUTO-SELECT vs BEST TRA ===')
display(seq_df)

out1 = BASE / 'scores' / 'stage3_tra_by_backend.csv'
out2 = BASE / 'scores' / 'stage3_autoselect_vs_best.csv'
tra_df.to_csv(out1, index=False)
seq_df.to_csv(out2, index=False)
print('Wrote', out1)
print('Wrote', out2)
print('Paste both CSVs back for the Stage-3 report — measured only.')

### What success looks like

1. On **SIM+ 02**, Cellpose TRA >> threshold TRA (threshold was 0.0).
2. Auto-select picks a backend whose TRA is near the best measured TRA.
3. GOWT1 DET/TRA rise under Cellpose vs threshold.

If auto-select misses best TRA, that is a measured finding — improve the selection rule, do not invent scores.